# California Housing — EDA и baseline

Регрессия: предсказание **медианной стоимости жилья** (`MedHouseVal`, $100k). То же направление, что и скрипты в `src/`, но в интерактивной форме для задания «сначала ноутбук, затем .py».

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

bunch = fetch_california_housing(as_frame=True)
df = bunch.frame
df.head()

In [ ]:
X = df.drop(columns=[bunch.target.name])
y = df[bunch.target.name]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("R2:", r2_score(y_test, pred), "MAE:", mean_absolute_error(y_test, pred))

Дальше перенос в репозиторий: см. `src/preprocess.py`, `src/train.py`, `dvc.yaml`, `api/main.py`.

## 1) Импорт библиотек и загрузка данных

Повторяем стиль учебного ноутбука: сначала знакомимся с данными, затем EDA, потом baseline и сравнение моделей.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

In [ ]:
bunch = fetch_california_housing(as_frame=True)
df = bunch.frame.copy()
TARGET_COL = bunch.target.name
FEATURE_COLS = [c for c in df.columns if c != TARGET_COL]

print("shape:", df.shape)
print("target:", TARGET_COL)
df.head()

## 2) Первичный анализ данных

Смотрим типы, пропуски, описательную статистику и распределение целевой переменной.

In [ ]:
df.info()

display(df.isna().sum().to_frame("missing"))
display(df.describe().T)

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df[TARGET_COL], bins=40, kde=True)
plt.title("Target distribution: MedHouseVal")
plt.show()

## 3) EDA: связи признаков с целевой переменной

In [ ]:
corr = df.corr(numeric_only=True)
plt.figure(figsize=(10, 7))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation heatmap")
plt.show()

corr_target = corr[TARGET_COL].sort_values(ascending=False)
display(corr_target.to_frame("corr_with_target"))

In [ ]:
sample_cols = ["MedInc", "AveRooms", "Latitude", "Longitude", TARGET_COL]
sns.pairplot(df[sample_cols].sample(1200, random_state=42), corner=True, diag_kind="kde")
plt.show()

## 4) Обучение baseline и сравнение моделей

Сравним линейную регрессию (простой baseline) и RandomForestRegressor.

In [ ]:
X = df[FEATURE_COLS]
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    "LinearRegression": LinearRegression(),
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=100,
        max_depth=12,
        random_state=42,
        n_jobs=-1,
    ),
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append(
        {
            "model": name,
            "R2": r2_score(y_test, pred),
            "MAE": mean_absolute_error(y_test, pred),
            "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        }
    )

results = pd.DataFrame(rows).sort_values("R2", ascending=False)
display(results)

In [ ]:
best_model = models["RandomForestRegressor"]
y_pred = best_model.predict(X_test)

plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, s=8, alpha=0.4)
lim = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lim, lim, "r--")
plt.xlabel("True MedHouseVal")
plt.ylabel("Predicted MedHouseVal")
plt.title("RandomForest: true vs predicted")
plt.show()

## 5) Выводы

- В данных нет пропусков, признаки числовые и готовы к моделированию.
- `MedInc` обычно самый сильный предиктор целевой переменной.
- `RandomForestRegressor` ожидаемо даёт лучшее качество, чем линейный baseline.
- Далее эта логика вынесена в production-скрипты: `src/preprocess.py`, `src/train.py`, `api/main.py`.

## 6) Полное обучение как в `src` (preprocess + train + сохранение артефактов)

Ниже — тот же сценарий, что в скриптах проекта:
1. подготовка директорий и путей,
2. сохранение raw/processed CSV,
3. train/test split,
4. чтение гиперпараметров из `config.ini`,
5. обучение `RandomForestRegressor`,
6. расчёт метрик,
7. сохранение `models/model.joblib`.

In [ ]:
from pathlib import Path
import configparser
import joblib

ROOT = Path.cwd().resolve()
RAW_DIR = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_NAME = "MedHouseVal"
FEATURE_NAMES = [
    "MedInc",
    "HouseAge",
    "AveRooms",
    "AveBedrms",
    "Population",
    "AveOccup",
    "Latitude",
    "Longitude",
]

raw_csv = RAW_DIR / "california_housing.csv"
if raw_csv.is_file():
    df_full = pd.read_csv(raw_csv)
else:
    bunch = fetch_california_housing(as_frame=True)
    df_full = bunch.frame
    df_full.to_csv(raw_csv, index=False)

X = df_full[FEATURE_NAMES]
y = df_full[TARGET_NAME]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train.to_csv(PROC_DIR / "X_train.csv", index=False)
X_test.to_csv(PROC_DIR / "X_test.csv", index=False)
y_train.to_csv(PROC_DIR / "y_train.csv", index=False)
y_test.to_csv(PROC_DIR / "y_test.csv", index=False)

print("Saved:")
print(" -", raw_csv)
print(" -", PROC_DIR / "X_train.csv")
print(" -", PROC_DIR / "X_test.csv")
print(" -", PROC_DIR / "y_train.csv")
print(" -", PROC_DIR / "y_test.csv")

In [ ]:
cfg = configparser.ConfigParser()
cfg_path = ROOT / "config.ini"
cfg.read(cfg_path, encoding="utf-8")

n_estimators = int(cfg.get("MODEL", "n_estimators", fallback="100"))
md = cfg.get("MODEL", "max_depth", fallback="12").strip().lower()
max_depth = None if md in ("", "none", "null") else int(md)
random_state = int(cfg.get("MODEL", "random_state", fallback="42"))
n_jobs = int(cfg.get("MODEL", "n_jobs", fallback="-1"))

model = RandomForestRegressor(
    n_estimators=n_estimators,
    max_depth=max_depth,
    random_state=random_state,
    n_jobs=n_jobs,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
metrics = {
    "r2": float(r2_score(y_test, y_pred)),
    "mae": float(mean_absolute_error(y_test, y_pred)),
}

model_path = MODELS_DIR / "model.joblib"
joblib.dump({"model": model, "feature_names": FEATURE_NAMES}, model_path)

print("Hyperparameters from config.ini:")
print({"n_estimators": n_estimators, "max_depth": max_depth, "random_state": random_state, "n_jobs": n_jobs})
print("Metrics:", metrics)
print("Saved model:", model_path)